In [ ]:
import os
import json
import asyncio
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from openai import AsyncOpenAI
from sklearn.model_selection import train_test_split
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type


# =========================================================
# CONFIGURAÇÃO
# =========================================================
@dataclass(frozen=True)
class ProviderConfig:
    name: str
    base_url: str
    api_key_env: str
    model: str


@dataclass
class AppConfig:
    input_path: Path = Path("/Users/jose-cleiton/dev/script_pncp/data/dataset/pncp_exemplos_2000.csv")
    base_data_dir: Path = Path("/Users/jose-cleiton/dev/script_pncp/data")

    text_cols: tuple[str, ...] = ("texto", "text", "Objeto", "objetoCompra")
    val_size: float = 0.2
    seed: int = 42

    provider_name: str = "gemini"
    batch_size: int = 15
    max_concurrent_tasks: int = 6
    max_retries: int = 5
    temperature: float = 0.1

    exclude_low_confidence: bool = True
    exclude_ambiguous: bool = True

    save_raw_csv: bool = True
    save_summary_json: bool = True

    run_id: str = field(default_factory=lambda: datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
    output_dir: Path = field(init=False)

    providers: Dict[str, ProviderConfig] = field(default_factory=lambda: {
        "openai": ProviderConfig(
            name="openai",
            base_url="",
            api_key_env="OPENAI_API_KEY",
            model="gpt-5.4-mini",
        ),
        "gemini": ProviderConfig(
            name="gemini",
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key_env="GEMINI_API_KEY",
            model="gemini-2.0-flash",
        ),
    })

    def __post_init__(self) -> None:
        self.output_dir = self.base_data_dir / "runs" / self.run_id
        self.output_dir.mkdir(parents=True, exist_ok=True)

    @property
    def provider(self) -> ProviderConfig:
        if self.provider_name not in self.providers:
            raise ValueError(f"Provider inválido: {self.provider_name}")
        return self.providers[self.provider_name]


@dataclass(frozen=True)
class OutputPaths:
    raw_csv: Path
    ambiguous_csv: Path
    full_csv: Path
    train_csv: Path
    validation_csv: Path
    summary_json: Path
    checkpoint_json: Path

    @staticmethod
    def from_config(config: AppConfig) -> "OutputPaths":
        return OutputPaths(
            raw_csv=config.output_dir / "classificacao_raw.csv",
            ambiguous_csv=config.output_dir / "ambiguous_review.csv",
            full_csv=config.output_dir / "dataset_bart_full.csv",
            train_csv=config.output_dir / "train.csv",
            validation_csv=config.output_dir / "validation.csv",
            summary_json=config.output_dir / "run_summary.json",
            checkpoint_json=config.output_dir / "progress.json",
        )


# =========================================================
# LOG
# =========================================================
class Logger:
    def info(self, message: str) -> None:
        print(message, flush=True)

    def error(self, message: str) -> None:
        print(message, flush=True)


# =========================================================
# PROMPTS
# =========================================================
class PromptFactory:
    OPENAI_PROMPT = """
Você é um classificador rigoroso de itens de licitação.

Objetivo:
Decidir se um item serve para o meu negócio.

Meu negócio vende soluções de:
1. Controle de acesso físico:
   - catracas, torniquetes, cancelas, portas automáticas, fechaduras eletrônicas/eletromagnéticas,
     controladoras de acesso, botoeiras, detectores de metais, leitores faciais, biometria facial.
2. Relógio de ponto e frequência:
   - relógio de ponto, REP, ponto biométrico, ponto facial, controle de frequência,
     bobina/papel para relógio de ponto, bastão de ronda.
3. Softwares dedicados:
   - software de controle de acesso, software de portaria, gestão de visitantes,
     software de ponto/frequência, software de acesso físico.
4. Marcas e linhas relevantes quando ligadas a acesso físico:
   - HID, Hikvision, iDFace.

Regras:
- Marque true somente quando o item estiver claramente dentro desse escopo.
- Se houver dúvida real, marque false.
- Não classifique como relevante itens genéricos de TI, RH, obras, logística, saúde ou trânsito.
- Responda exatamente um resultado por texto.
- Preserve o campo "texto" exatamente como recebido.
- Em "motivo", explique em 1 ou 2 frases curtas.
- Use categorias curtas e estáveis, como:
  controle_acesso, controle_ponto, biometria_facial, catraca, fechadura,
  portaria, acessorio_ponto, fora_escopo, ambiguo.

Exemplos de false:
- controle de acesso à rede, firewall, VPN, Active Directory
- porta de madeira, porta de vidro comum
- motor de veículo
- catraca de carga/caminhão
- software de RH generalista
- prontuário eletrônico
- software de multas
- monitoramento de frota
- estação de solda
"""

    GEMINI_PROMPT = """
Você é um triador B2B extremamente conservador para licitações.

Tarefa:
Decidir se cada item pertence ao nosso portfólio comercial.

ACEITAR (serve_para_mim = true) somente quando o item estiver claramente ligado a:
A) Controle de acesso físico:
- catracas, torniquetes, cancelas, portas automáticas, fechaduras eletrônicas/eletromagnéticas
- controladoras de acesso, botoeiras, detectores de metais, leitores faciais
- biometria facial, reconhecimento facial para entrada/saída física de pessoas
B) Ponto e frequência:
- relógio de ponto, REP, ponto biométrico, ponto facial
- controle/registro de frequência
- bobina e papel para relógio de ponto
- bastão de ronda
C) Softwares dedicados:
- software de controle de acesso físico
- software de portaria/visitantes
- software de ponto/frequência
D) Marcas do domínio:
- HID, Hikvision, iDFace, quando vinculadas a acesso físico/ponto

REJEITAR (serve_para_mim = false) em casos como:
- acesso lógico/TI: firewall, VPN, rede, login, diretório, Active Directory
- porta comum de obra/carpintaria
- motor que não seja de portão
- catraca de carga/logística
- software genérico de RH, ERP, folha, prontuário
- monitoramento de frota, rastreamento veicular, telemetria
- software de multas/trânsito
- estação de solda, itens industriais sem relação com acesso/ponto

Regras obrigatórias:
1. Seja conservador.
2. Na dúvida real, marque false.
3. Preserve "texto" exatamente como recebido.
4. Retorne exatamente um resultado por item.
5. Em "motivo", explique de forma curta e objetiva.
6. Use categorias curtas e estáveis, como:
   controle_acesso, controle_ponto, biometria_facial, catraca, fechadura,
   portaria, acessorio_ponto, fora_escopo, ambiguo.
"""

    SCHEMA = {
        "type": "object",
        "properties": {
            "resultados": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "texto": {"type": "string"},
                        "motivo": {"type": "string"},
                        "serve_para_mim": {"type": "boolean"},
                        "categoria": {"type": "string"},
                        "confianca": {
                            "type": "string",
                            "enum": ["alta", "media", "baixa"],
                        },
                    },
                    "required": ["texto", "motivo", "serve_para_mim", "categoria", "confianca"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["resultados"],
        "additionalProperties": False,
    }

    def get_system_prompt(self, provider_name: str) -> str:
        return self.OPENAI_PROMPT if provider_name == "openai" else self.GEMINI_PROMPT

    def get_schema(self) -> Dict[str, Any]:
        return self.SCHEMA


# =========================================================
# CLIENTE
# =========================================================
class ClientFactory:
    def __init__(self, config: AppConfig):
        self.config = config

    def create(self) -> AsyncOpenAI:
        provider = self.config.provider
        api_key = os.environ.get(provider.api_key_env)
        if not api_key:
            raise ValueError(f"Falta a variável de ambiente {provider.api_key_env}")

        base_url = provider.base_url or None
        return AsyncOpenAI(api_key=api_key, base_url=base_url)


# =========================================================
# DATASET
# =========================================================
class DatasetLoader:
    def __init__(self, config: AppConfig):
        self.config = config

    def load_texts(self) -> List[str]:
        df_raw = pd.read_csv(self.config.input_path)
        text_col = self._find_text_column(df_raw)
        df = self._clean_dataframe(df_raw, text_col)
        return df["text"].tolist()

    def _find_text_column(self, df: pd.DataFrame) -> str:
        for col in self.config.text_cols:
            if col in df.columns:
                return col
        raise ValueError(f"Nenhuma coluna de texto encontrada entre: {self.config.text_cols}")

    def _clean_dataframe(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        cleaned = df[[text_col]].copy()
        cleaned = cleaned.rename(columns={text_col: "text"})
        cleaned["text"] = cleaned["text"].apply(self._normalize_text)
        cleaned = cleaned[cleaned["text"] != ""].drop_duplicates(subset=["text"]).reset_index(drop=True)
        return cleaned

    @staticmethod
    def _normalize_text(value: Any) -> str:
        if pd.isna(value):
            return ""
        text = " ".join(str(value).strip().split())
        return "" if text.lower() in {"nan", "none", "null"} else text


class BatchBuilder:
    def __init__(self, batch_size: int):
        self.batch_size = batch_size

    def build(self, items: List[str]) -> List[List[str]]:
        return [items[i:i + self.batch_size] for i in range(0, len(items), self.batch_size)]


# =========================================================
# CHECKPOINT
# =========================================================
class CheckpointRepository:
    def __init__(self, path: Path):
        self.path = path
        self.lock = asyncio.Lock()
        self.data = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            return json.loads(self.path.read_text(encoding="utf-8"))
        return {"completed_batches": [], "results_by_batch": {}}

    def is_completed(self, batch_idx: int) -> bool:
        return batch_idx in self.data["completed_batches"]

    async def save_batch(self, batch_idx: int, results: List[dict]) -> None:
        async with self.lock:
            if batch_idx not in self.data["completed_batches"]:
                self.data["completed_batches"].append(batch_idx)
            self.data["results_by_batch"][str(batch_idx)] = results
            self._write_atomically()

    def get_all_results(self) -> List[dict]:
        results: List[dict] = []
        for key in sorted(self.data["results_by_batch"].keys(), key=int):
            results.extend(self.data["results_by_batch"][key])
        return results

    def _write_atomically(self) -> None:
        temp_path = self.path.with_suffix(".tmp")
        temp_path.write_text(json.dumps(self.data, ensure_ascii=False, indent=2), encoding="utf-8")
        temp_path.replace(self.path)


# =========================================================
# CLASSIFICADOR
# =========================================================
class AbstractClassifier(ABC):
    @abstractmethod
    async def classify(self, texts: List[str]) -> List[dict]:
        pass


class LlmClassifier(AbstractClassifier):
    def __init__(
        self,
        client: AsyncOpenAI,
        config: AppConfig,
        prompt_factory: PromptFactory,
    ):
        self.client = client
        self.config = config
        self.prompt_factory = prompt_factory

    @retry(
        stop=stop_after_attempt(5),
        wait=wait_exponential(multiplier=1, min=2, max=30),
        retry=retry_if_exception_type(Exception),
        reraise=True,
    )
    async def classify(self, texts: List[str]) -> List[dict]:
        system_prompt = self.prompt_factory.get_system_prompt(self.config.provider_name)
        schema = self.prompt_factory.get_schema()

        prompt_user = {
            "instrucoes": [
                "Classifique cada item.",
                "Retorne exatamente um resultado por texto.",
                "Mantenha o texto original no campo 'texto'.",
                "Se houver dúvida real, marque false.",
            ],
            "itens": texts,
        }

        response = await self.client.chat.completions.create(
            model=self.config.provider.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(prompt_user, ensure_ascii=False)},
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "classificacao_itens_negocio",
                    "schema": schema,
                    "strict": True,
                },
            },
            temperature=self.config.temperature,
        )

        content = response.choices[0].message.content
        parsed = json.loads(content)
        results = parsed.get("resultados", [])

        if len(results) != len(texts):
            raise ValueError(
                f"Quantidade divergente: modelo retornou {len(results)} itens para {len(texts)} textos."
            )

        return [self._normalize_result(original, item) for original, item in zip(texts, results)]

    @staticmethod
    def _normalize_result(original_text: str, item: Dict[str, Any]) -> Dict[str, Any]:
        return {
            "texto": original_text,
            "motivo": item.get("motivo", ""),
            "serve_para_mim": item.get("serve_para_mim"),
            "categoria": item.get("categoria", ""),
            "confianca": item.get("confianca", "baixa"),
        }


# =========================================================
# EXECUÇÃO DE LOTES
# =========================================================
class BatchProcessor:
    def __init__(
        self,
        classifier: AbstractClassifier,
        checkpoint_repository: CheckpointRepository,
        logger: Logger,
        max_concurrent_tasks: int,
    ):
        self.classifier = classifier
        self.checkpoint_repository = checkpoint_repository
        self.logger = logger
        self.semaphore = asyncio.Semaphore(max_concurrent_tasks)

    async def process_all(self, batches: List[List[str]]) -> None:
        tasks = [
            self._process_single_batch(idx, batch)
            for idx, batch in enumerate(batches)
        ]
        await asyncio.gather(*tasks)

    async def _process_single_batch(self, batch_idx: int, texts: List[str]) -> None:
        if self.checkpoint_repository.is_completed(batch_idx):
            return

        async with self.semaphore:
            try:
                results = await self.classifier.classify(texts)
                await self.checkpoint_repository.save_batch(batch_idx, results)
                self.logger.info(f"[OK] lote {batch_idx} concluído.")
            except Exception as exc:
                self.logger.error(f"[ERRO] lote {batch_idx} falhou após retries: {exc}")
                fallback = [
                    {
                        "texto": text,
                        "motivo": f"Erro de classificação: {exc}",
                        "serve_para_mim": False,
                        "categoria": "erro_classificacao",
                        "confianca": "baixa",
                    }
                    for text in texts
                ]
                await self.checkpoint_repository.save_batch(batch_idx, fallback)


# =========================================================
# PÓS-PROCESSAMENTO
# =========================================================
class ResultsPostProcessor:
    def __init__(self, config: AppConfig, output_paths: OutputPaths, logger: Logger):
        self.config = config
        self.output_paths = output_paths
        self.logger = logger

    def process(self, raw_results: List[dict]) -> None:
        df = self._build_raw_dataframe(raw_results)

        if self.config.save_raw_csv:
            df.to_csv(self.output_paths.raw_csv, index=False, encoding="utf-8-sig")
            df[df["ambiguous_review"]].to_csv(
                self.output_paths.ambiguous_csv,
                index=False,
                encoding="utf-8-sig",
            )

        final_df = self._build_training_dataframe(df)
        final_df.to_csv(self.output_paths.full_csv, index=False, encoding="utf-8-sig")

        if final_df.empty:
            raise RuntimeError("Dataset final ficou vazio.")
        if len(final_df["label"].unique()) < 2:
            raise RuntimeError("Dataset final ficou com apenas uma classe.")

        train_df, val_df = train_test_split(
            final_df,
            test_size=self.config.val_size,
            random_state=self.config.seed,
            stratify=final_df["label"],
        )

        train_df.to_csv(self.output_paths.train_csv, index=False, encoding="utf-8-sig")
        val_df.to_csv(self.output_paths.validation_csv, index=False, encoding="utf-8-sig")

        if self.config.save_summary_json:
            self._save_summary(df, final_df, train_df, val_df)

        self.logger.info("\n✅ Concluído.")
        self.logger.info(f"Run ID: {self.config.run_id}")
        self.logger.info(f"Dataset final: {self.output_paths.full_csv}")
        self.logger.info(f"Train: {self.output_paths.train_csv}")
        self.logger.info(f"Validation: {self.output_paths.validation_csv}")
        self.logger.info(f"Ambiguous review: {self.output_paths.ambiguous_csv}")
        self.logger.info(f"Distribuição final:\n{final_df['label'].value_counts()}")

    def _build_raw_dataframe(self, raw_results: List[dict]) -> pd.DataFrame:
        df = pd.DataFrame(raw_results)

        if "texto" not in df.columns:
            raise ValueError("Coluna 'texto' ausente no resultado final.")

        df["text"] = df["texto"].apply(self._normalize_text)
        df = df[df["text"] != ""].copy()
        df["label"] = df["serve_para_mim"].apply(self._bool_to_label)
        df["ambiguous_review"] = df.apply(self._is_ambiguous_row, axis=1)
        return df

    def _build_training_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        train_source = df.copy()

        if self.config.exclude_low_confidence:
            train_source = train_source[train_source["confianca"] != "baixa"].copy()

        if self.config.exclude_ambiguous:
            train_source = train_source[~train_source["ambiguous_review"]].copy()

        train_source = train_source[train_source["label"].isin([0, 1])].copy()
        train_source["label"] = train_source["label"].astype(int)

        return train_source[["text", "label"]].drop_duplicates().reset_index(drop=True)

    def _save_summary(
        self,
        raw_df: pd.DataFrame,
        final_df: pd.DataFrame,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
    ) -> None:
        summary = {
            "run_id": self.config.run_id,
            "provider": self.config.provider_name,
            "model": self.config.provider.model,
            "input_path": str(self.config.input_path),
            "output_dir": str(self.config.output_dir),
            "batch_size": self.config.batch_size,
            "max_concurrent_tasks": self.config.max_concurrent_tasks,
            "temperature": self.config.temperature,
            "exclude_low_confidence": self.config.exclude_low_confidence,
            "exclude_ambiguous": self.config.exclude_ambiguous,
            "rows_raw_results": int(len(raw_df)),
            "rows_ambiguous": int(raw_df["ambiguous_review"].sum()),
            "rows_final_dataset": int(len(final_df)),
            "rows_train": int(len(train_df)),
            "rows_validation": int(len(val_df)),
            "label_distribution_full": {
                str(k): int(v) for k, v in final_df["label"].value_counts().to_dict().items()
            },
            "label_distribution_train": {
                str(k): int(v) for k, v in train_df["label"].value_counts().to_dict().items()
            },
            "label_distribution_validation": {
                str(k): int(v) for k, v in val_df["label"].value_counts().to_dict().items()
            },
        }
        self.output_paths.summary_json.write_text(
            json.dumps(summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    @staticmethod
    def _normalize_text(value: Any) -> str:
        if pd.isna(value):
            return ""
        text = " ".join(str(value).strip().split())
        return "" if text.lower() in {"nan", "none", "null"} else text

    @staticmethod
    def _bool_to_label(value: Any) -> Optional[int]:
        if value is True:
            return 1
        if value is False:
            return 0
        return None

    @staticmethod
    def _is_ambiguous_row(row: pd.Series) -> bool:
        motivo = str(row.get("motivo", "")).lower()
        categoria = str(row.get("categoria", "")).lower()
        confianca = str(row.get("confianca", "")).lower()

        ambiguity_terms = [
            "ambígu", "ambigu", "dúvida", "duvida", "incerto", "incerta",
            "depende", "não está claro", "nao esta claro", "genérico", "generico",
        ]

        if categoria == "ambiguo":
            return True
        if confianca == "baixa":
            return True
        return any(term in motivo for term in ambiguity_terms)


# =========================================================
# ORQUESTRADOR
# =========================================================
class PipelineApp:
    def __init__(self, config: AppConfig):
        self.config = config
        self.output_paths = OutputPaths.from_config(config)
        self.logger = Logger()

        self.prompt_factory = PromptFactory()
        self.client_factory = ClientFactory(config)
        self.dataset_loader = DatasetLoader(config)
        self.batch_builder = BatchBuilder(config.batch_size)
        self.checkpoint_repository = CheckpointRepository(self.output_paths.checkpoint_json)
        self.post_processor = ResultsPostProcessor(config, self.output_paths, self.logger)

    async def run(self) -> None:
        client = self.client_factory.create()
        classifier = LlmClassifier(client, self.config, self.prompt_factory)
        batch_processor = BatchProcessor(
            classifier=classifier,
            checkpoint_repository=self.checkpoint_repository,
            logger=self.logger,
            max_concurrent_tasks=self.config.max_concurrent_tasks,
        )

        texts = self.dataset_loader.load_texts()
        batches = self.batch_builder.build(texts)

        self.logger.info(f"Run ID: {self.config.run_id}")
        self.logger.info(f"Output dir: {self.config.output_dir}")
        self.logger.info(f"Provider: {self.config.provider_name}")
        self.logger.info(f"Model: {self.config.provider.model}")
        self.logger.info(f"Itens válidos: {len(texts)}")
        self.logger.info(f"Lotes: {len(batches)}")
        self.logger.info(f"Concorrência máxima: {self.config.max_concurrent_tasks}")

        await batch_processor.process_all(batches)

        raw_results = self.checkpoint_repository.get_all_results()
        self.post_processor.process(raw_results)


# =========================================================
# MAIN
# =========================================================
async def main() -> None:
    config = AppConfig()
    app = PipelineApp(config)
    await app.run()

# if __name__ == "__main__":
#     asyncio.run(main())

In [7]:
# No Jupyter use await diretamente — asyncio.run() não funciona em event loop já ativo
await main()


Run ID: 2026-03-21_10-23-32
Output dir: /Users/jose-cleiton/dev/script_pncp/data/runs/2026-03-21_10-23-32
Provider: gemini
Model: gemini-2.0-flash
Itens válidos: 1699
Lotes: 114
Concorrência máxima: 6
Output dir: /Users/jose-cleiton/dev/script_pncp/data/runs/2026-03-21_10-23-32
Provider: gemini
Model: gemini-2.0-flash
Itens válidos: 1699
Lotes: 114
Concorrência máxima: 6
[OK] lote 4 concluído.
[OK] lote 4 concluído.
[OK] lote 3 concluído.
[OK] lote 3 concluído.
[OK] lote 1 concluído.
[OK] lote 1 concluído.
[OK] lote 5 concluído.
[OK] lote 5 concluído.
[OK] lote 2 concluído.
[OK] lote 0 concluído.
[OK] lote 2 concluído.
[OK] lote 0 concluído.
[OK] lote 6 concluído.
[OK] lote 6 concluído.
[OK] lote 7 concluído.
[OK] lote 7 concluído.
[OK] lote 8 concluído.
[OK] lote 8 concluído.
[OK] lote 9 concluído.
[OK] lote 9 concluído.
[OK] lote 10 concluído.
[OK] lote 10 concluído.
[OK] lote 11 concluído.
[OK] lote 11 concluído.
[OK] lote 12 concluído.
[OK] lote 13 concluído.
[OK] lote 12 concluído